# Qwen3 model-size J-space study

This notebook runs one model at a time and writes resumable artifacts to Google Drive. It keeps standard HumanEval/GSM8K performance, oracle local-Jacobian controllability, and BIPIA security results in separate directories. The local J-space view is a target-logit sensitivity measurement, **not** a decoder of private thoughts.

Use an L4 for 4B/8B/14B and an A100 40 GB for 32B. All checkpoints load as frozen NF4 with BF16 compute.

In [ ]:
# A Cursor notebook uses the remote Colab filesystem, not your local checkout.
# This branch must be committed and pushed before Colab can clone it.
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/ethanncyb/jspace-research.git'
REPO_BRANCH = 'codex/qwen-model-size-test'
REPO_ROOT = Path('/content/jspace-research')

if not (REPO_ROOT / '.git').is_dir():
    clone = subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
        text=True, capture_output=True,
    )
    if clone.returncode != 0:
        raise RuntimeError(
            f'Could not clone {REPO_BRANCH!r} from {REPO_URL}. '            'Commit and push the branch first, then restart the Colab runtime.\n'
            f'git said: {clone.stderr.strip()}'
        )

required = [REPO_ROOT / 'pyproject.toml', REPO_ROOT / 'jlens' / 'study.py']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError(
        'The Colab checkout does not contain the study code: ' + ', '.join(missing) +
        '. Commit/push the current branch and start with a fresh runtime.'
    )

%cd $REPO_ROOT
%pip install -q -e '.[study]'

# Fail here, with context, rather than at the experiment cell.
subprocess.run(
    [sys.executable, '-c', 'import jlens, jlens.study; print(\"jlens import OK:\", jlens.__file__)'],
    check=True,
)

# BIPIA supplies the distributable email/table/code data. Its full package
# dependencies include vLLM/DeepSpeed, so jlens imports only the builders.
BIPIA_ROOT = '/content/BIPIA'
BIPIA_COMMIT = 'a004b69ec0dd446e0afd461d98cb5e96e120a5d0'
if not os.path.isdir(BIPIA_ROOT):
    subprocess.run(['git', 'clone', 'https://github.com/microsoft/BIPIA.git', BIPIA_ROOT], check=True)
subprocess.run(['git', '-C', BIPIA_ROOT, 'checkout', '--detach', BIPIA_COMMIT], check=True)
assert os.path.isfile(os.path.join(BIPIA_ROOT, 'bipia', 'data', '__init__.py'))

## 1. Select exactly one model

Change this one cell between sessions. `smoke` validates the pipeline; `full` runs all 164 HumanEval and 1,319 GSM8K cases, plus 64 steering cases per dataset and 64 BIPIA pairs per task/split.

In [ ]:
ACTIVE_MODEL = 'qwen3-4b'
# ACTIVE_MODEL = 'qwen3-8b'
# ACTIVE_MODEL = 'qwen3-14b'
# ACTIVE_MODEL = 'qwen3-32b'
RUN_PROFILE = 'smoke'  # change to 'full' after the smoke run succeeds

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, torch
from pathlib import Path
import jlens
from jlens.study import StudyConfig, RunManifest, JsonlStore, preflight, load_quantized_model

config = StudyConfig(active_model=ACTIVE_MODEL, profile=RUN_PROFILE)
environment = preflight(config)
config.run_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(environment, indent=2))

## 2. Load NF4 model and exercise an activation gradient

This fails before the long run if the model layout, NF4 backward path, BF16 values, or selected GPU is incompatible.

In [ ]:
hf_model, tokenizer, model = load_quantized_model(config)
probe_ids = tokenizer('A short gradient preflight', return_tensors='pt').input_ids.to(model.input_device)
probe_target = tokenizer(' test', add_special_tokens=False).input_ids[0]
probe_local = jlens.compute_local_jacobian(model, probe_ids, target_token_id=probe_target, layers=[model.n_layers // 2])
assert torch.isfinite(probe_local.sensitivity()).all()
model_commit = getattr(hf_model.config, '_commit_hash', None)
tokenizer_commit = tokenizer.init_kwargs.get('_commit_hash')
bipia_commit = subprocess.run(['git', '-C', BIPIA_ROOT, 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
manifest = RunManifest.create(config, environment=environment, model_commit=model_commit, tokenizer_commit=tokenizer_commit, dataset_revisions={'humaneval': config.humaneval_revision, 'gsm8k': config.gsm8k_revision, 'BIPIA': bipia_commit})
manifest.write(config.run_dir / 'manifest.json')
print(model, 'gradient preflight passed')

## 3. Standard behavioral benchmarks

These are the only rows used for HumanEval pass@1 and GSM8K exact match. Qwen3 thinking and final-answer text are stored separately. Generated HumanEval code executes only in a timed, resource-limited subprocess in this disposable Colab runtime.

In [ ]:
from jlens.study import load_behavior_cases, run_behavioral_benchmark

benchmark_dir = config.run_dir / 'benchmarks'
benchmark_store = JsonlStore(benchmark_dir / 'observations.jsonl', config_hash=config.config_hash)
behavior_cases = load_behavior_cases(config.run_profile, humaneval_revision=config.humaneval_revision, gsm8k_revision=config.gsm8k_revision)
behavior_rows = run_behavioral_benchmark(
    hf_model, tokenizer, behavior_cases, benchmark_store, config.generation,
    progress=lambda done, total, case: print(f'{done + 1}/{total} {case.dataset} {case.case_id}'),
)
print('behavior rows:', len(behavior_rows))

## 4. Oracle local-Jacobian controllability

This is not benchmark accuracy. Each case teacher-forces a canonical reference prefix, differentiates one gold token, and compares the exact local-gradient direction with a seeded matched-norm random direction at 25%, 50%, and 75% depth over α = 0, 0.025, 0.05, 0.1, 0.2, 0.4.

In [ ]:
from jlens.study import run_steering_case, append_steering_rows, write_local_jspace_page

n_steering = config.run_profile.steering_cases_per_dataset
steering_cases = (
    jlens.load_humaneval_cases(tokenizer, n_examples=n_steering, seed=config.seed)
    + jlens.load_gsm8k_cases(tokenizer, n_examples=n_steering, seed=config.seed)
)
steering_dir = config.run_dir / 'steering'
steering_store = JsonlStore(steering_dir / 'observations.jsonl', config_hash=config.config_hash, id_field='observation_id')
preview_store = JsonlStore(config.run_dir / 'jspace' / 'generation_previews.jsonl', config_hash=config.config_hash)
heatmap_counts = {'humaneval': 0, 'gsm8k': 0}
for index, case in enumerate(steering_cases):
    prefix = f'{case.dataset}:{case.case_id}:'
    expected = 1 + len(jlens.relative_layers(model.n_layers, config.depth_fractions)) * (len(config.strengths) - 1) * 2
    case_complete = sum(str(row['observation_id']).startswith(prefix) for row in steering_store.rows()) == expected
    wants_visual = heatmap_counts[case.dataset] < config.run_profile.heatmap_cases_per_dataset
    safe_case_id = case.case_id.replace('/', '_')
    heatmap_path = config.run_dir / 'jspace' / f'{case.dataset}-{safe_case_id}.html'
    preview_id = f'{case.dataset}:{case.case_id}'
    visual_complete = heatmap_path.exists() and preview_store.contains(preview_id)
    if case_complete and (not wants_visual or visual_complete):
        if wants_visual:
            heatmap_counts[case.dataset] += 1
        continue
    print(f'{index + 1}/{len(steering_cases)} {case.dataset} {case.case_id}')
    local, rows = run_steering_case(model, dataset=case.dataset, case_id=case.case_id, input_ids=case.input_ids, target_token_id=case.target_token_id, target_text=case.target_text, strengths=config.strengths, depth_fractions=config.depth_fractions, random_seed=config.seed)
    append_steering_rows(steering_store, rows)
    if wants_visual:
        if not heatmap_path.exists():
            write_local_jspace_page(heatmap_path, local, rows, tokenizer, title=f'{config.model.model_id} — {case.dataset} {case.case_id}')
        if not preview_store.contains(preview_id):
            preview_layer = jlens.relative_layers(model.n_layers, config.depth_fractions)[1]
            preview = jlens.generate_local_comparison(hf_model, model, local, layer=preview_layer, strength=0.4, max_new_tokens=32, random_seed=config.seed)
            preview_store.append({'case_id': preview_id, 'dataset': case.dataset, 'target_text': case.target_text, 'layer': preview_layer, 'strength': 0.4, **preview})
        heatmap_counts[case.dataset] += 1
steering_rows = steering_store.rows()
print('steering observations:', len(steering_rows))

## 5. BIPIA recognition, residual probe, and susceptibility

The three outputs remain separate: explicit BENIGN/INJECTION self-report, PCA-128 logistic probes at eight normalized-depth checkpoints, and harmless-canary attack success plus clean utility. The existing circuit breaker and continual-learning prototype are not activated.

In [ ]:
from jlens.security_study import (load_bipia_examples, split_train_validation, load_or_collect_features, train_fixed_rank_probes, evaluate_fixed_rank_probes, run_security_behavior, summarize_security_behavior, FixedRankLayerProbe)
from jlens.study import write_csv_summary

limit = config.run_profile.security_cases
train_all = load_bipia_examples(BIPIA_ROOT, split='train', limit_pairs_per_task=limit, seed=config.seed)
train_all = split_train_validation(train_all, seed=config.seed)
security_train = [row for row in train_all if row.split == 'train']
security_validation = [row for row in train_all if row.split == 'validation']
security_test = load_bipia_examples(BIPIA_ROOT, split='test', limit_pairs_per_task=limit, seed=config.seed)

security_dir = config.run_dir / 'security'
train_features = load_or_collect_features(security_dir / 'train_features.pt', model, tokenizer, security_train, model_id=config.model.model_id)
validation_features = load_or_collect_features(security_dir / 'validation_features.pt', model, tokenizer, security_validation, model_id=config.model.model_id)
test_features = load_or_collect_features(security_dir / 'test_features.pt', model, tokenizer, security_test, model_id=config.model.model_id)
probe_path = security_dir / 'probe.pt'
if probe_path.exists():
    probe = FixedRankLayerProbe.load(probe_path, expected_model_id=config.model.model_id, expected_layers=sorted(train_features))
else:
    probe = train_fixed_rank_probes(train_features, [row.label for row in security_train], validation_features, [row.label for row in security_validation], rank=128, seed=config.seed)
    probe.save(probe_path, model_id=config.model.model_id)
probe_rows = evaluate_fixed_rank_probes(probe, test_features, [row.label for row in security_test])
write_csv_summary(security_dir / 'probe_per_layer.csv', probe_rows)

security_store = JsonlStore(security_dir / 'behavior.jsonl', config_hash=config.config_hash)
security_rows = run_security_behavior(hf_model, tokenizer, security_test, security_store, config.generation)
security_summary = summarize_security_behavior(security_rows)
print(json.dumps(security_summary, indent=2))

## 6. Export this model and compare completed models

JSONL remains authoritative for resume. Parquet and CSV are analysis copies. The cross-model report is explicitly exploratory because four sizes do not establish a scaling law.

In [ ]:
from jlens.analysis import build_model_summary, compare_models, model_summary_row, write_comparison_markdown, write_json, write_parquet
from jlens.study import write_csv_summary

write_parquet(benchmark_dir / 'observations.parquet', behavior_rows)
write_parquet(steering_dir / 'observations.parquet', steering_rows)
write_parquet(security_dir / 'behavior.parquet', security_rows)
summary = build_model_summary(model=config.active_model, parameters_b=config.model.parameters_b, behavior_rows=behavior_rows, steering_rows=steering_rows, security_summary=security_summary, probe_rows=probe_rows)
summary_path = write_json(config.run_dir / 'summary.json', summary)
write_csv_summary(config.run_dir / 'summary.csv', [model_summary_row(summary)])

all_summaries = []
for path in Path(config.output_root).glob(f'{config.experiment_hash}/*/summary.json'):
    all_summaries.append(json.loads(path.read_text()))
if len(all_summaries) > 1:
    report = compare_models(all_summaries)
    write_json(Path(config.output_root) / config.experiment_hash / 'comparison.json', report)
    write_comparison_markdown(Path(config.output_root) / config.experiment_hash / 'comparison.md', report)
    write_csv_summary(Path(config.output_root) / config.experiment_hash / 'comparison.csv', [model_summary_row(row) for row in report['models']])
    print(json.dumps(report['spearman_vs_log_parameters'], indent=2))
print('saved', summary_path)

In [ ]:
# Safe to run more than once, including after a partially failed smoke run.
for name in ('probe_local', 'probe', 'train_features', 'validation_features', 'test_features', 'model', 'hf_model', 'tokenizer'):
    globals().pop(name, None)
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print('model released; change ACTIVE_MODEL and restart from section 1')